# 信贷逾期风险分析：业务理解与数据概览

本阶段用于明确业务问题、标签口径、字段角色与数据边界，并完成后续分析所需的首次数据审计。

## 1. 业务问题

信贷风险管理需要在业务规模与资产质量之间取得平衡。本项目关注以下问题：

1. 数据中的整体坏客户水平如何？
2. 哪些客户特征能够稳定地区分未来严重逾期风险？
3. 哪些组合客群需要更严格的准入、额度或风险监控策略？
4. 不同准入规则会如何影响通过率、坏客户捕获率和好客户误伤率？

## 2. 路径与分析口径

- 分析对象：公开数据中的个人信贷客户记录；
- 目标变量：`SeriousDlqin2yrs`；
- 坏客户：目标变量等于 1，表示未来两年发生严重逾期或财务困境；
- 好客户：目标变量等于 0；
- 主要指标：样本量、坏客户数、坏客户率、通过率、坏客户捕获率和好客户误伤率。

In [4]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="Microsoft YaHei")

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "output"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

for path in [PROCESSED_DATA_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

DATA_PATH = RAW_DATA_DIR / "cs-training.csv"

PROJECT_ROOT, DATA_PATH

(WindowsPath('e:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis'),
 WindowsPath('e:/DataAnalysis/DataAnalysisRoad/Projects/01_Credit_Risk_Analysis/data/raw/cs-training.csv'))

## 3. 数据读取

In [6]:
raw_df = pd.read_csv(DATA_PATH)
raw_df.head()
# 字段重命名表：原始字段名 -> 简洁字段名
column_mapping = {
    "timestamp": "timestamp",
    "id": "id",
    "SeriousDlqin2yrs": "is_bad_customer",
    "RevolvingUtilizationOfUnsecuredLines": "credit_utilization",
    "age": "age",
    "NumberOfTime30-59DaysPastDueNotWorse": "past_due_30_59_count",
    "DebtRatio": "debt_ratio",
    "MonthlyIncome": "monthly_income",
    "NumberOfOpenCreditLinesAndLoans": "open_credit_count",
    "NumberOfTimes90DaysLate": "past_due_90_count",
    "NumberRealEstateLoansOrLines": "real_estate_loan_count",
    "NumberOfTime60-89DaysPastDueNotWorse": "past_due_60_89_count",
    "NumberOfDependents": "dependent_count",
}

# rename() 默认返回新表，不会修改原始 raw_df
renamed_df = raw_df.rename(columns=column_mapping)

renamed_df.head()

,timestamp,id,is_bad_customer,credit_utilization,age,past_due_30_59_count,debt_ratio,monthly_income,open_credit_count,past_due_90_count,real_estate_loan_count,past_due_60_89_count,dependent_count
0,1970-01-01 00:00:00,1,1,0.7661,45,2,0.8030,"9,120.0000",13,0,6,0,2.0000
1,1970-01-01 00:00:00,2,0,0.9572,40,0,0.1219,"2,600.0000",4,0,0,0,1.0000
2,1970-01-01 00:00:00,3,0,0.6582,38,1,0.0851,"3,042.0000",2,1,0,0,0.0000
3,1970-01-01 00:00:00,4,0,0.2338,30,0,0.0360,"3,300.0000",5,0,0,0,0.0000
4,1970-01-01 00:00:00,5,0,0.9072,49,1,0.0249,"63,588.0000",7,0,1,0,0.0000


In [13]:
data_overview = pd.DataFrame(
    {
        "指标": [
            "样本数",
            "字段数",
            "重复行数",
            "缺失值总数",
            "含缺失值字段数",
            "坏客户数",
            "坏客户率",
        ],
        "结果": [
            len(renamed_df),
            renamed_df.shape[1],
            renamed_df.duplicated().sum(),
            renamed_df.isna().sum().sum(),
            (renamed_df.isna().sum() > 0).sum(),
            renamed_df["is_bad_customer"].sum(),
            renamed_df["is_bad_customer"].mean(),
        ],
    }
)

print(data_overview)
renamed_df.info()

        指标           结果
0      样本数 150,000.0000
1      字段数      13.0000
2     重复行数       0.0000
3    缺失值总数  33,655.0000
4  含缺失值字段数       2.0000
5     坏客户数  10,026.0000
6     坏客户率       0.0668
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 13 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   timestamp               150000 non-null  str    
 1   id                      150000 non-null  int64  
 2   is_bad_customer         150000 non-null  int64  
 3   credit_utilization      150000 non-null  float64
 4   age                     150000 non-null  int64  
 5   past_due_30_59_count    150000 non-null  int64  
 6   debt_ratio              150000 non-null  float64
 7   monthly_income          120269 non-null  float64
 8   open_credit_count       150000 non-null  int64  
 9   past_due_90_count       150000 non-null  int64  
 10  real_estate_loan_count  150000 non-null  int64  
 11  p

## 4. 字段检查

公开镜像比常见 Kaggle 文件多出 `timestamp` 和 `id` 两列。使用前需要确认它们是否具有真实业务含义。

In [ ]:
metadata_check = pd.DataFrame(
    {
        "字段": ["timestamp", "id"],
        "唯一值数量": [renamed_df["timestamp"].nunique(), renamed_df["id"].nunique()],
        "最小值": [renamed_df["timestamp"].min(), renamed_df["id"].min()],
        "最大值": [renamed_df["timestamp"].max(), renamed_df["id"].max()],
    }
)
metadata_check

,字段,唯一值数量,最小值,最大值
0,timestamp,1,1970-01-01 00:00:00,1970-01-01 00:00:00
1,id,150000,1,150000


**处理判断：** `timestamp` 全部相同，是镜像附加的占位字段，不代表申请时间，应从分析数据中删除；`id` 为顺序编号，仅用于记录定位，不作为风险特征。

In [15]:
analysis_df = renamed_df.drop(columns="timestamp").copy()
analysis_df.head()
field_summary = pd.DataFrame(
    {
        "字段": analysis_df.columns,
        "数据类型": analysis_df.dtypes.astype(str).values,
        "非空数量": analysis_df.notna().sum().values,
        "缺失数量": analysis_df.isna().sum().values,
        "唯一值数量": analysis_df.nunique(dropna=True).values,
    }
)
field_summary

,字段,数据类型,非空数量,缺失数量,唯一值数量
0,id,int64,150000,0,150000
1,is_bad_customer,int64,150000,0,2
2,credit_utilization,float64,150000,0,125728
3,age,int64,150000,0,86
4,past_due_30_59_count,int64,150000,0,16
5,debt_ratio,float64,150000,0,114194
6,monthly_income,float64,120269,29731,13594
7,open_credit_count,int64,150000,0,58
8,past_due_90_count,int64,150000,0,19
9,real_estate_loan_count,int64,150000,0,28


## 5. 目标变量分布

In [6]:
target_col = "SeriousDlqin2yrs"

target_summary = (
    analysis_df[target_col]
    .value_counts()
    .rename_axis("客户类型编码")
    .to_frame("客户数")
    .assign(客户占比=lambda x: x["客户数"] / x["客户数"].sum())
    .reset_index()
)
target_summary["客户类型"] = target_summary["客户类型编码"].map({0: "好客户", 1: "坏客户"})
target_summary[["客户类型编码", "客户类型", "客户数", "客户占比"]]

,客户类型编码,客户类型,客户数,客户占比
0,0,好客户,139974,0.9332
1,1,坏客户,10026,0.0668


In [7]:
overall_bad_rate = analysis_df[target_col].mean()
print(f"整体坏客户率：{overall_bad_rate:.2%}")

整体坏客户率：6.68%


整体坏客户率用于描述样本中的风险基准。由于好坏客户数量差异明显，后续制定规则时不能只看准确率，还需要关注坏客户捕获率、好客户误伤率以及不同风险层的样本占比。

## 6. 缺失值概览

In [8]:
missing_summary = (
    analysis_df.isna()
    .agg(["sum", "mean"])
    .T
    .rename(columns={"sum": "缺失数量", "mean": "缺失率"})
    .query("缺失数量 > 0")
    .sort_values("缺失率", ascending=False)
)
missing_summary

,缺失数量,缺失率
MonthlyIncome,"29,731.0000",0.1982
NumberOfDependents,"3,924.0000",0.0262


收入和家属数量存在缺失。缺失本身可能包含风险信息，因此在比较缺失客群与非缺失客群的坏客户率之前，不直接采用均值填充或删除记录。

## 7. 数值字段初步审计

In [9]:
analysis_df.drop(columns="id").describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
SeriousDlqin2yrs,"150,000.0000",0.0668,0.2497,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,1.0000
RevolvingUtilizationOfUnsecuredLines,"150,000.0000",6.0484,249.7554,0.0000,0.0000,0.0000,0.0299,0.1542,0.5590,1.0000,1.0930,"50,708.0000"
age,"150,000.0000",52.2952,14.7719,0.0000,24.0000,29.0000,41.0000,52.0000,63.0000,78.0000,87.0000,109.0000
NumberOfTime30-59DaysPastDueNotWorse,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,4.0000,98.0000
DebtRatio,"150,000.0000",353.0051,"2,037.8185",0.0000,0.0000,0.0043,0.1751,0.3665,0.8683,"2,449.0000","4,979.0400","329,664.0000"
MonthlyIncome,"120,269.0000","6,670.2212","14,384.6742",0.0000,0.0000,"1,300.0000","3,400.0000","5,400.0000","8,249.0000","14,587.6000","25,000.0000","3,008,750.0000"
NumberOfOpenCreditLinesAndLoans,"150,000.0000",8.4528,5.1460,0.0000,0.0000,2.0000,5.0000,8.0000,11.0000,18.0000,24.0000,58.0000
NumberOfTimes90DaysLate,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,3.0000,98.0000
NumberRealEstateLoansOrLines,"150,000.0000",1.0182,1.1298,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,3.0000,4.0000,54.0000
NumberOfTime60-89DaysPastDueNotWorse,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,2.0000,98.0000


In [10]:
audit_items = pd.DataFrame(
    {
        "检查项": [
            "年龄为0",
            "月收入缺失",
            "家属数量缺失",
            "额度使用率大于1",
            "30至59天逾期次数为96或98",
            "60至89天逾期次数为96或98",
            "90天以上逾期次数为96或98",
        ],
        "记录数": [
            analysis_df["age"].eq(0).sum(),
            analysis_df["MonthlyIncome"].isna().sum(),
            analysis_df["NumberOfDependents"].isna().sum(),
            analysis_df["RevolvingUtilizationOfUnsecuredLines"].gt(1).sum(),
            analysis_df["NumberOfTime30-59DaysPastDueNotWorse"].isin([96, 98]).sum(),
            analysis_df["NumberOfTime60-89DaysPastDueNotWorse"].isin([96, 98]).sum(),
            analysis_df["NumberOfTimes90DaysLate"].isin([96, 98]).sum(),
        ],
    }
)
audit_items

,检查项,记录数
0,年龄为0,1
1,月收入缺失,29731
2,家属数量缺失,3924
3,额度使用率大于1,3321
4,30至59天逾期次数为96或98,269
5,60至89天逾期次数为96或98,269
6,90天以上逾期次数为96或98,269


本阶段只识别异常候选值，不直接删除。额度使用率大于 1 可能代表超额使用；逾期次数中的 96、98 需要结合三个字段的共同分布和坏客户率进一步判断。

## 8. 待验证的业务假设

1. 历史逾期次数越多，未来发生严重逾期的风险越高；
2. 无担保循环信贷额度使用率较高的客户具有更高坏客户率；
3. 收入缺失客户与收入非缺失客户存在明显风险差异；
4. 年龄与坏客户率可能存在非线性关系，年轻客群风险可能更高；
5. 高负债率与低收入或收入缺失同时出现时，风险差异可能更加明显。

以上内容均为分析假设，需要通过分组样本量、坏客户率以及开发集和验证集的一致性进行检验。

## 9. 项目边界

- 数据没有真实申请时间，不能使用占位 `timestamp` 进行时间切分；
- 数据缺少贷款金额、利率和资金成本，不能计算真实利润与风险成本；
- 数据缺少完整账期表现，不能开展严格的 Vintage、MOB 或迁徙分析；
- 后续规则模拟反映样本内风险区分效果，不代表线上策略的因果收益；
- 变量与风险之间的关系应表述为相关关系，不直接解释为因果关系。

## 10. 初步结论

- 数据包含 150,000 条客户记录，目标变量完整，无重复行；
- 样本整体坏客户率约为 6.68%，好坏客户分布不均衡；
- 月收入和家属数量存在缺失，需要比较缺失本身的风险差异；
- 年龄为0、逾期次数特殊值以及额度使用率极端值需要进入下一阶段审计；
- `timestamp` 不具备业务含义，`id` 仅作为记录编号。